# Stage 1 — 09A: A7 hard-slice & temporal-uncertainty audit

This notebook **does not train**. It starts from the original DLC-only A7
`V-JEPA 2.1-B` checkpoint and asks a different question from notebook 05:

> If 1/3/5-clip mean aggregation is already saturated on the 104-video DLC val,
> which videos / metadata slices are fragile even when their final label is correct?

Notebook 05 already showed that 1/3/5 deterministic clips all reach 1.0 on the
fixed validation split. 09A therefore runs **five clips once**, keeps clip-level
probabilities, and analyses centre/mean/median/trimmed/logit/max/top-2
aggregation, probability margin, temporal variance/range, clip disagreement,
and device/condition/document-type/group slices.

Final A7-compatible evaluation here uses **FP32 with no autocast**, matching the
safe submission path after the earlier attention dtype issue.


## 1. Setup


In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path

REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage1-sangchun"

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    print("Not running in Google Colab; Drive mount skipped.")

if IN_COLAB:
    REPO_ROOT = Path("/content/Blackbox-Detection")
    if not (REPO_ROOT / ".git").is_dir():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "--single-branch", REPO_URL, str(REPO_ROOT),
        ], check=True)
    else:
        current_branch = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if current_branch != BRANCH:
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", BRANCH], check=True)
        dirty = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if dirty:
            print("WARNING: local repo has changes; git pull skipped.")
        else:
            subprocess.run(
                ["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", BRANCH],
                check=True,
            )
else:
    REPO_ROOT = Path.cwd().resolve()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "pyproject.toml").is_file():
        raise FileNotFoundError("Run this notebook inside Blackbox-Detection repository.")

os.chdir(REPO_ROOT)

# Keep Colab's binary scientific stack intact. Install only the Stage 1 extras
# and then this repository editable with --no-deps, matching the current notebooks.
COLAB_EXTRAS = [
    "av>=15,<17", "timm==1.0.15", "fvcore==0.1.5.post20221221",
    "iopath==0.1.10", "yacs==0.1.8", "einops==0.8.1",
    "omegaconf==2.3.0", "hydra-core==1.3.2", "easydict==1.13",
]
if IN_COLAB:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed", *COLAB_EXTRAS,
    ], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO_ROOT)
], check=True)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

from blackbox_detection.utils import load_checkpoint, seed_everything

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATASET_ROOT = DRIVE_PROJECT_ROOT / "DATASET"
DLC_ROOT = DATASET_ROOT / "DLC-2021"
DLC_SPLIT_CSV = DLC_ROOT / "dlc_split.csv"
OUTPUT_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "stage1"
CONFIG_DIR = REPO_ROOT / "configs" / "stage1"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    required = {"DLC_ROOT": DLC_ROOT, "DLC_SPLIT_CSV": DLC_SPLIT_CSV}
    missing = [f"{k}: {v}" for k, v in required.items() if not v.exists()]
    if missing:
        raise FileNotFoundError("Missing required Drive paths:\n  " + "\n  ".join(missing))

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
print("repo   :", REPO_ROOT)
print("branch :", BRANCH)
print("commit :", GIT_COMMIT)
print("torch  :", torch.__version__)
print("cuda   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu    :", torch.cuda.get_device_name(0))


## 2. Load exact A7 V-JEPA 2.1-B configuration


In [ ]:
from blackbox_detection.stage1.dataset import Stage1VideoDataset, build_dataloader, video_batch_adapter
from blackbox_detection.stage1.evaluator import (
    AggregationConfig, Stage1Evaluator, aggregate_unit_predictions,
    evaluate_predictions, probabilities_to_labels, save_predictions,
)
from blackbox_detection.stage1.models import build_stage1_model, count_parameters
from blackbox_detection.stage1.sampling import build_clip_sampler
from blackbox_detection.stage1.transforms import ClipAugmentConfig, build_video_transforms

MODEL_NAME = "vjepa2_1_b"
CONFIG = yaml.safe_load((CONFIG_DIR / "vjepa2_1_b.yaml").read_text(encoding="utf-8"))
assert CONFIG["model"]["name"] == MODEL_NAME
SEED = int(CONFIG["train"]["seed"])
seed_everything(SEED, deterministic=False)

VJEPA_SOURCE_ROOT = Path("/content/vjepa2")
VJEPA_CKPT_DIR = DRIVE_PROJECT_ROOT / "pretrained" / "vjepa2"
VJEPA_CKPT_DIR.mkdir(parents=True, exist_ok=True)
VJEPA_CKPT = VJEPA_CKPT_DIR / "vjepa2_1_vitb_dist_vitG_384.pt"
if not (VJEPA_SOURCE_ROOT / ".git").is_dir():
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/facebookresearch/vjepa2.git",
        str(VJEPA_SOURCE_ROOT),
    ], check=True)
if not VJEPA_CKPT.is_file():
    subprocess.run([
        "wget", "-O", str(VJEPA_CKPT),
        "https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt",
    ], check=True)
params = dict(CONFIG["model"]["params"])
params["source_root"] = str(VJEPA_SOURCE_ROOT)
params["checkpoint_path"] = str(VJEPA_CKPT)
params["allow_download"] = False
print("V-JEPA source :", params["source_root"])
print("V-JEPA ckpt   :", params["checkpoint_path"])
print("arch          :", params["arch"])
print("runtime input :", params["input_frames"], "frames @", params["input_size"])


## 3. Paths and fixed DLC validation manifest


In [ ]:
A7_RUN_DIR = OUTPUT_ROOT / "dlc" / MODEL_NAME
A7_CKPT = A7_RUN_DIR / "best.pt"
RUN_DIR = OUTPUT_ROOT / "09a_a7_hard_validation"
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not A7_CKPT.is_file():
    raise FileNotFoundError(f"A7 best.pt not found: {A7_CKPT}")
print("A7 checkpoint:", A7_CKPT)
print("09A output   :", RUN_DIR)


In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}

def _normalize_rel_text(value: str) -> str:
    return str(value).replace("\\", "/").strip().lstrip("./")

def _build_dlc_video_index() -> dict[tuple[str, str], str]:
    index = {}
    counts = {}
    for source in ("or", "re"):
        clips_root = DLC_ROOT / source / "clips_video"
        if not clips_root.is_dir():
            raise FileNotFoundError(f"DLC clips directory not found: {clips_root}")
        count = 0
        for path in clips_root.rglob("*"):
            if not path.is_file() or path.suffix.lower() not in VIDEO_EXTENSIONS:
                continue
            rel = path.relative_to(clips_root).with_suffix("").as_posix()
            key = (source, _normalize_rel_text(rel))
            if key in index and index[key] != str(path):
                raise ValueError(f"Duplicate DLC video key: {key}")
            index[key] = str(path)
            count += 1
        counts[source] = count
    print("indexed DLC videos:", counts)
    return index

DLC_VIDEO_INDEX = _build_dlc_video_index()

def _resolve_dlc_video(source: str, clip_id: str) -> str:
    source = str(source).strip().lower()
    clip_id = _normalize_rel_text(clip_id)
    key = (source, clip_id)
    if key in DLC_VIDEO_INDEX:
        return DLC_VIDEO_INDEX[key]
    matches = []
    for (src, rel), path in DLC_VIDEO_INDEX.items():
        if src != source:
            continue
        if rel.startswith(clip_id + "/") or rel.endswith("/" + clip_id) or rel == clip_id:
            matches.append(path)
    if len(matches) == 1:
        return matches[0]
    leaf = Path(clip_id).name
    matches = [
        path for (src, rel), path in DLC_VIDEO_INDEX.items()
        if src == source and Path(rel).name == leaf
    ]
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f"Cannot resolve DLC source={source!r}, clip_id={clip_id!r}")

def load_dlc_manifest(split_name: str) -> pd.DataFrame:
    raw = pd.read_csv(DLC_SPLIT_CSV).copy()
    required = {
        "clip_id", "class", "source", "document_type", "document_id",
        "group", "split", "device", "condition",
    }
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"dlc_split.csv missing columns: {missing}")
    raw["split"] = raw["split"].astype(str).str.strip().str.lower()
    raw["source"] = raw["source"].astype(str).str.strip().str.lower()
    raw["class"] = raw["class"].astype(str).str.strip().str.lower()
    frame = raw.loc[raw["split"].eq(split_name)].copy()
    if frame.empty:
        raise ValueError(f"No DLC rows for split={split_name!r}")
    frame["label"] = frame["class"].map({"original": "ORIGINAL", "rerecorded": "RERECORDED"})
    if frame["label"].isna().any():
        raise ValueError("Unexpected DLC class value found.")
    frame["video_id"] = "dlc__" + frame["clip_id"].astype(str).str.replace("/", "__", regex=False)
    frame["dataset"] = "dlc2021"
    frame["scene_type"] = "document"
    frame["video_path"] = [
        _resolve_dlc_video(source, clip_id)
        for source, clip_id in zip(frame["source"], frame["clip_id"])
    ]
    keep = [
        "video_path", "label", "video_id", "dataset", "scene_type",
        "clip_id", "source", "document_type", "document_id", "group",
        "device", "condition",
    ]
    out = frame[keep].reset_index(drop=True)
    missing_paths = [p for p in out["video_path"] if not Path(p).is_file()]
    if missing_paths:
        raise FileNotFoundError(
            f"{split_name}: {len(missing_paths)} missing videos; examples={missing_paths[:3]}"
        )
    return out


In [ ]:
val_df = load_dlc_manifest("val")
print("validation videos:", len(val_df))
display(pd.crosstab(val_df["label"], val_df["device"], margins=True))
display(pd.crosstab(val_df["label"], val_df["condition"], margins=True))


## 4. Restore A7 and verify architecture


In [ ]:
model = build_stage1_model(
    MODEL_NAME,
    finetune_mode=CONFIG["model"]["finetune_mode"],
    unfreeze_last_n=int(CONFIG["model"]["unfreeze_last_n"]),
    **params,
)
load_checkpoint(A7_CKPT, model=model, map_location="cpu", restore_rng_state=False)
print("blocks       :", len(model.blocks))
print("feature dim  :", model.feature_dim)
print("parameters   :", count_parameters(model))
print("preprocessing:", model.preprocessing())
print("load report  :", json.dumps(getattr(model, "load_report", {}), indent=2, default=str))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device).eval()


## 5. Decode five deterministic clips once


In [ ]:
video_cfg = CONFIG["data"]
aug_cfg = CONFIG["augmentation"]
pre = model.preprocessing()
_, val_transform = build_video_transforms(
    crop_size=int(pre["input_size"]),
    mean=tuple(pre["mean"]),
    std=tuple(pre["std"]),
    train_config=ClipAugmentConfig(
        crop_size=int(pre["input_size"]),
        scale_range=tuple(aug_cfg["scale_range"]),
        ratio_range=tuple(aug_cfg["ratio_range"]),
        hflip_prob=float(aug_cfg["hflip_prob"]),
        brightness=float(aug_cfg["brightness"]),
        contrast=float(aug_cfg["contrast"]),
        perspective_prob=float(aug_cfg["perspective_prob"]),
        perspective_scale=float(aug_cfg["perspective_scale"]),
    ),
)
NUM_CLIPS = 5
dataset = Stage1VideoDataset(
    val_df,
    clip_sampler=build_clip_sampler(
        train=False,
        num_frames=int(video_cfg["num_frames"]),
        val_stride=int(video_cfg["val_stride"]),
        num_clips=NUM_CLIPS,
    ),
    transform=val_transform,
    on_error="zero",
    deterministic=True,
)
loader = build_dataloader(
    dataset, batch_size=1, shuffle=False, num_workers=2, seed=SEED,
    persistent_workers=True,
)
evaluator = Stage1Evaluator(
    model, video_batch_adapter(), device=device, amp=False,
    aggregation=AggregationConfig(frame_method="mean", video_method="mean"),
)
started = time.perf_counter()
units = evaluator.predict_units(loader)
five_clip_elapsed = time.perf_counter() - started
units.to_csv(RUN_DIR / "a7_5clip_units.csv", index=False)
print(f"5-clip unit rows: {len(units)} | runtime: {five_clip_elapsed / 60:.2f} min")

# Exact A7 anchor: num_clips=1 uses DeterministicClipSampler's original
# max_start // 2 centre rule. The middle slot of a 5-clip sampler is usually
# close but is not guaranteed bit-identical for every odd max_start.
center_dataset = Stage1VideoDataset(
    val_df,
    clip_sampler=build_clip_sampler(
        train=False,
        num_frames=int(video_cfg["num_frames"]),
        val_stride=int(video_cfg["val_stride"]),
        num_clips=1,
    ),
    transform=val_transform,
    on_error="zero",
    deterministic=True,
)
center_loader = build_dataloader(
    center_dataset, batch_size=1, shuffle=False, num_workers=2, seed=SEED,
    persistent_workers=True,
)
center_started = time.perf_counter()
centre_units = evaluator.predict_units(center_loader)
center_elapsed = time.perf_counter() - center_started
centre_units.to_csv(RUN_DIR / "a7_center_units.csv", index=False)
elapsed = five_clip_elapsed + center_elapsed
print(f"exact centre rows: {len(centre_units)} | runtime: {center_elapsed / 60:.2f} min")
display(units.head(10))


## 6. Centre clip and alternative aggregation


In [ ]:
from blackbox_detection.utils.metrics import stage1_score

META_COLS = [
    "video_id", "clip_id", "source", "document_type", "document_id",
    "group", "device", "condition",
]
meta = val_df[META_COLS].drop_duplicates("video_id")

def _score_video_table(frame: pd.DataFrame, method: str):
    result = evaluate_predictions(frame, threshold=0.5, search_threshold=False)
    pred = result.predictions.merge(meta, on="video_id", how="left", validate="one_to_one")
    pred["method"] = method
    return result, pred

tables = {}
rows = []
centre_video = centre_units[["video_id", "label", "dataset", "prob_rerecorded"]].copy()
centre_video["prob_original"] = 1.0 - centre_video["prob_rerecorded"]
result, pred = _score_video_table(centre_video, "center")
tables["center"] = pred
rows.append({"method":"center", "macro_f1_at_0.5":float(result.macro_f1_at_default)})

for method in ("mean", "median", "trimmed_mean", "logit_mean", "max", "min"):
    video = aggregate_unit_predictions(
        units, aggregation=AggregationConfig(frame_method="mean", video_method=method)
    )
    result, pred = _score_video_table(video, method)
    tables[method] = pred
    rows.append({"method":method, "macro_f1_at_0.5":float(result.macro_f1_at_default)})

def topk_mean(values, k=2):
    values = np.sort(np.asarray(values, dtype=np.float64))
    return float(values[-min(k, len(values)):].mean())

top2 = (
    units.loc[units["valid"].astype(bool)]
    .groupby(["video_id", "label", "dataset"], as_index=False)["prob_rerecorded"]
    .agg(topk_mean)
)
top2["prob_original"] = 1.0 - top2["prob_rerecorded"]
result, pred = _score_video_table(top2, "top2_mean")
tables["top2_mean"] = pred
rows.append({"method":"top2_mean", "macro_f1_at_0.5":float(result.macro_f1_at_default)})
summary = pd.DataFrame(rows).sort_values(["macro_f1_at_0.5", "method"], ascending=[False, True])
summary.to_csv(RUN_DIR / "aggregation_summary.csv", index=False)
display(summary)


## 7. Temporal uncertainty and hard cases


In [ ]:
valid_units = units.loc[units["valid"].astype(bool) & units["prob_rerecorded"].notna()].copy()
stats = (
    valid_units.groupby(["video_id", "label", "dataset"], as_index=False)
    .agg(
        p_mean=("prob_rerecorded", "mean"),
        p_std=("prob_rerecorded", "std"),
        p_min=("prob_rerecorded", "min"),
        p_max=("prob_rerecorded", "max"),
        p_median=("prob_rerecorded", "median"),
    )
)
stats["p_std"] = stats["p_std"].fillna(0.0)
stats["p_range"] = stats["p_max"] - stats["p_min"]
stats["margin"] = (stats["p_mean"] - 0.5).abs()
clip_labels = valid_units.assign(
    clip_pred=np.where(valid_units["prob_rerecorded"].ge(0.5), "RERECORDED", "ORIGINAL")
)
disagreement = clip_labels.groupby("video_id")["clip_pred"].nunique().rename("num_clip_labels").reset_index()
stats = stats.merge(disagreement, on="video_id", how="left")
stats["clip_label_disagreement"] = stats["num_clip_labels"].gt(1)
stats = stats.merge(meta, on="video_id", how="left", validate="one_to_one")
stats["rank_margin"] = stats["margin"].rank(method="min", ascending=True)
stats["rank_range"] = stats["p_range"].rank(method="min", ascending=False)
stats["hardness_score"] = stats["rank_margin"] + stats["rank_range"]
hard_cases = stats.sort_values(
    ["clip_label_disagreement", "hardness_score"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
hard_cases.to_csv(RUN_DIR / "hard_cases.csv", index=False)
display(hard_cases.head(30))


## 8. Metadata slice audit


In [ ]:
def slice_report(pred: pd.DataFrame, column: str, min_rows: int = 4) -> pd.DataFrame:
    if column in pred.columns:
        merged = pred.copy()
    else:
        merged = pred.merge(
            val_df[["video_id", column]].drop_duplicates("video_id"),
            on="video_id", how="left", validate="one_to_one",
        )
    out = []
    for value, group in merged.groupby(column, dropna=False):
        if len(group) < min_rows:
            continue
        payload = {
            "slice_column": column,
            "slice_value": str(value),
            "num_videos": int(len(group)),
            "num_original": int(group["label"].eq("ORIGINAL").sum()),
            "num_rerecorded": int(group["label"].eq("RERECORDED").sum()),
            "mean_margin": float((group["prob_rerecorded"] - 0.5).abs().mean()),
        }
        if group["label"].nunique() == 2:
            payload["macro_f1_at_0.5"] = float(stage1_score(
                group["label"], probabilities_to_labels(group["prob_rerecorded"], 0.5)
            ))
        else:
            payload["macro_f1_at_0.5"] = np.nan
        out.append(payload)
    return pd.DataFrame(out)

slice_frames = []
for column in ("device", "condition", "document_type", "group"):
    f = slice_report(tables["center"], column)
    if len(f):
        slice_frames.append(f)
slice_summary = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
if len(slice_summary):
    slice_summary = slice_summary.sort_values(
        ["macro_f1_at_0.5", "mean_margin", "num_videos"],
        ascending=[True, True, False], na_position="last", kind="mergesort",
    )
slice_summary.to_csv(RUN_DIR / "slice_summary.csv", index=False)
display(slice_summary.head(50))


## 9. Save A7 centre predictions and summary


In [ ]:
a7_center = tables["center"].copy()
drop_cols = [c for c in META_COLS[1:] + ["method"] if c in a7_center.columns]
save_predictions(a7_center.drop(columns=drop_cols), RUN_DIR / "a7_center_val_predictions.csv")
audit = {
    "experiment": "09a_a7_hard_validation",
    "source_checkpoint": str(A7_CKPT),
    "git_commit": GIT_COMMIT,
    "num_clips": NUM_CLIPS,
    "fp32_no_autocast": True,
    "runtime_seconds": float(elapsed),
    "aggregation_summary": summary.to_dict(orient="records"),
    "num_temporal_label_disagreements": int(hard_cases["clip_label_disagreement"].sum()),
    "median_probability_range": float(hard_cases["p_range"].median()),
    "max_probability_range": float(hard_cases["p_range"].max()),
    "min_center_margin": float((a7_center["prob_rerecorded"] - 0.5).abs().min()),
}
(RUN_DIR / "summary.json").write_text(json.dumps(audit, indent=2, default=str), encoding="utf-8")
print(json.dumps(audit, indent=2))
print("saved to:", RUN_DIR)


## Interpretation

Do not pick an aggregation only because the same 104-video validation split
remains at 1.0. The useful outputs are the probability margins, temporal
instability, and metadata slices. They become stress-test diagnostics for
09B–11.
